In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import run_pipeline

In [2]:
summary_full = run_pipeline(mode="full", fetch=False)

2026-04-24 12:08:23,693 [INFO] pipeline — === Pipeline started ===
2026-04-24 12:08:23,694 [INFO] pipeline — Mode: full
2026-04-24 12:08:23,759 [INFO] src.database — Connected to DuckDB: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Weather-Prediction-Project\data\caspian_weather.duckdb
2026-04-24 12:08:23,761 [INFO] src.database —   Schema 'raw' ready.
2026-04-24 12:08:23,762 [INFO] src.database —   Schema 'staging' ready.
2026-04-24 12:08:23,762 [INFO] src.database —   Schema 'analytics' ready.
2026-04-24 12:08:23,768 [INFO] src.database —   Created raw.weather_daily
2026-04-24 12:08:23,771 [INFO] src.database —   Created raw.visibility_daily
2026-04-24 12:08:23,773 [INFO] src.database —   Created raw.forecast_7day
2026-04-24 12:08:23,774 [INFO] pipeline — Skipping API fetch. Using existing raw CSV files.
2026-04-24 12:08:23,774 [INFO] pipeline — Loading raw CSV files into DuckDB.
2026-04-24 12:08:23,832 [INFO] src.database —   Loaded aktau_historical_2019_20

In [3]:
summary_full["quality_checks"]

,check_name,stage,status,details
0,Row count,raw,PASS,"10,960 rows in raw.weather_daily"
1,Freshness,raw,WARNING,"Latest raw date is 2024-12-31, which is 479 da..."
2,Null ratio,staging,PASS,city: 0.00% nulls
3,Null ratio,staging,PASS,date: 0.00% nulls
4,Null ratio,staging,PASS,temperature_2m_max: 0.00% nulls
5,Null ratio,staging,PASS,temperature_2m_min: 0.00% nulls
6,Null ratio,staging,PASS,temperature_2m_mean: 0.00% nulls
7,Null ratio,staging,PASS,apparent_temperature_mean: 0.00% nulls
8,Null ratio,staging,PASS,wind_speed_10m_max: 0.00% nulls
9,Null ratio,staging,PASS,wind_gusts_10m_max: 0.00% nulls


In [4]:
summary_incremental = run_pipeline(mode="incremental", fetch=False)

2026-04-24 12:08:37,077 [INFO] pipeline — === Pipeline started ===
2026-04-24 12:08:37,077 [INFO] pipeline — Mode: incremental
2026-04-24 12:08:37,104 [INFO] src.database — Connected to DuckDB: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Weather-Prediction-Project\data\caspian_weather.duckdb
2026-04-24 12:08:37,105 [INFO] src.database —   Schema 'raw' ready.
2026-04-24 12:08:37,106 [INFO] src.database —   Schema 'staging' ready.
2026-04-24 12:08:37,107 [INFO] src.database —   Schema 'analytics' ready.
2026-04-24 12:08:37,108 [INFO] pipeline — Skipping API fetch. Using existing raw CSV files.
2026-04-24 12:08:37,109 [INFO] pipeline — Loading raw CSV files into DuckDB.
2026-04-24 12:08:37,151 [INFO] src.database —   Loaded aktau_historical_2019_2024.csv → raw.weather_daily
2026-04-24 12:08:37,190 [INFO] src.database —   Loaded anzali_historical_2019_2024.csv → raw.weather_daily
2026-04-24 12:08:37,231 [INFO] src.database —   Loaded baku_historical_2019_2024.csv

In [5]:
summary_incremental["quality_checks"]

,check_name,stage,status,details
0,Row count,raw,PASS,"10,960 rows in raw.weather_daily"
1,Freshness,raw,WARNING,"Latest raw date is 2024-12-31, which is 479 da..."
2,Null ratio,staging,PASS,city: 0.00% nulls
3,Null ratio,staging,PASS,date: 0.00% nulls
4,Null ratio,staging,PASS,temperature_2m_max: 0.00% nulls
5,Null ratio,staging,PASS,temperature_2m_min: 0.00% nulls
6,Null ratio,staging,PASS,temperature_2m_mean: 0.00% nulls
7,Null ratio,staging,PASS,apparent_temperature_mean: 0.00% nulls
8,Null ratio,staging,PASS,wind_speed_10m_max: 0.00% nulls
9,Null ratio,staging,PASS,wind_gusts_10m_max: 0.00% nulls


## Pipeline Architecture

```text
┌──────────────────────┐
│ Open-Meteo APIs      │
│ Weather + Visibility │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ src/ingestion.py     │
│ Fetch + save raw CSV │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ DuckDB raw schema    │
│ raw.weather_daily    │
│ raw.visibility_daily │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ src/cleaning.py      │
│ Missing values       │
│ Outlier flags        │
│ Date validation      │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ DuckDB staging schema│
│ staging.weather_daily│
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ src/features.py      │
│ Rolling features     │
│ Seasonal features    │
│ Risk labels          │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ DuckDB analytics     │
│ daily_enriched       │
│ monthly_summary      │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────┐
│ src/quality_checks.py│
│ Pass / Warning / Fail│
└──────────────────────┘